# Wan2.1 Cinematic Audio Studio
### Image → cinematic motion → spoken dialogue → lip-sync → narration + ambience → MP4

This is an audio-augmented workflow. Wan2.1 VACE generates the moving scene; Edge TTS creates spoken audio; MuseTalk 1.5 synchronizes the character's mouth to dialogue; FFmpeg mixes dialogue, optional narration, and an optional ambience/music track.

**Before running:** In Google Colab choose **Runtime → Change runtime type → GPU**. A GPU is required. Free GPU availability and quotas are controlled by Google. This pipeline downloads several large models and can take a long time.

**Important:** Wan2.1 itself does not generate synchronized speech audio in this workflow. Lip-sync works best when the source video has a clear, visible, front-facing face. The generated MP4 is AI-generated; use voices and likenesses only with appropriate permission.


In [ ]:
import os, sys, subprocess, json, textwrap, shutil, time
from pathlib import Path
import torch
print('Python:', sys.version)
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime required. In Colab choose Runtime → Change runtime type → T4 GPU (or another GPU), save, reconnect, then Runtime → Run all.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
print('FFmpeg ready.')


In [ ]:
from google.colab import files
from PIL import Image
WORK = Path('/content')
print('Upload the scene image you want to animate.')
up = files.upload()
if not up: raise RuntimeError('No scene image uploaded.')
uploaded_name = next(iter(up))
# Save the returned upload bytes explicitly to /content, regardless of current working directory.
IMAGE_PATH = str(WORK / Path(uploaded_name).name)
with open(IMAGE_PATH, 'wb') as image_file:
    image_file.write(up[uploaded_name])
if not Path(IMAGE_PATH).is_file():
    raise FileNotFoundError(f'Upload was not saved correctly: {IMAGE_PATH}')
display(Image.open(IMAGE_PATH).convert('RGB'))
print('Scene image saved at:', IMAGE_PATH)


## Create the scene and sound brief
Enter the visual action separately from the exact words spoken by the character. Narration and ambience are optional.

In [ ]:
VISUAL_PROMPT = input('VISUAL PROMPT — what happens and how should the camera move? ').strip()
DIALOGUE = input('CHARACTER DIALOGUE — exact words the person in the image should speak (required for lip-sync): ').strip()
NARRATION = input('OPTIONAL NARRATION / VOICE-OVER (press Enter to skip): ').strip()
if not VISUAL_PROMPT: raise ValueError('Visual prompt is required.')
if not DIALOGUE: raise ValueError('Dialogue is required for lip-sync.')
print('Voice options: en-NG-AbeoNeural, en-NG-EzinneNeural, en-US-GuyNeural, en-US-JennyNeural, en-GB-RyanNeural, en-GB-SoniaNeural')
DIALOGUE_VOICE = input('Dialogue voice (default en-NG-AbeoNeural): ').strip() or 'en-NG-AbeoNeural'
NARRATION_VOICE = input('Narration voice (default en-NG-EzinneNeural): ').strip() or 'en-NG-EzinneNeural'
print('Optional: upload one ambience/music/sound-effects audio file. Press Cancel in the file picker to skip.')
try:
    amb = files.upload()
    AMBIENCE_PATH = str(WORK / next(iter(amb))) if amb else None
except Exception:
    AMBIENCE_PATH = None
print('Ambience:', AMBIENCE_PATH or 'none')


In [ ]:
# Generate speech audio. Edge TTS is online and needs an internet connection; no API key is required.
subprocess.run([sys.executable,'-m','pip','install','-q','edge-tts'],check=True)
import asyncio, edge_tts
async def make_tts(text, voice, out):
    communicate = edge_tts.Communicate(text=text, voice=voice)
    await communicate.save(out)
asyncio.run(make_tts(DIALOGUE,DIALOGUE_VOICE,'/content/dialogue.mp3'))
if NARRATION:
    asyncio.run(make_tts(NARRATION,NARRATION_VOICE,'/content/narration.mp3'))
print('Speech audio generated.')


## Install Wan2.1 and generate the moving scene
This uses the official VACE 1.3B reference-to-video pipeline at 480p.

In [ ]:
req = (REPO/'requirements.txt').read_text().splitlines()
filtered = [x for x in req if x.strip() and not x.lstrip().startswith('#') and 'flash_attn' not in x.lower()]
Path('/content/wan_requirements.txt').write_text('\n'.join(filtered)+'\n')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','/content/wan_requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub'],check=True)
from huggingface_hub import snapshot_download
WAN_MODEL='/content/Wan2.1-VACE-1.3B'
snapshot_download(repo_id='Wan-AI/Wan2.1-VACE-1.3B',local_dir=WAN_MODEL)
print('Wan2.1 VACE weights ready.')


In [ ]:
WAN_VIDEO='/content/wan_scene_silent.mp4'
cmd=[sys.executable,'generate.py','--task','vace-1.3B','--size','832*480','--ckpt_dir',WAN_MODEL,'--src_ref_images',IMAGE_PATH,'--prompt',VISUAL_PROMPT,'--frame_num','81','--offload_model','True','--t5_cpu','--save_file',WAN_VIDEO]
print('Generating visual scene. This can take a while...')
subprocess.run(cmd,cwd=str(REPO),check=True)
assert Path(WAN_VIDEO).exists(), 'Wan2.1 did not produce the expected MP4.'
print('Silent scene generated:',WAN_VIDEO)


## Install MuseTalk 1.5 for character lip-sync
MuseTalk uses a separate Python 3.10 environment to avoid conflicting with Wan2.1's dependencies. This setup downloads its model weights.

In [ ]:
subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH'] += ':/root/.local/bin'
subprocess.run(['uv','python','install','3.10'],check=True)
MUSE=Path('/content/MuseTalk')
if not MUSE.exists(): subprocess.run(['git','clone','https://github.com/TMElyralab/MuseTalk.git',str(MUSE)],check=True)
MUSE_ENV='/content/musetalk-venv'
subprocess.run(['uv','venv','--python','3.10',MUSE_ENV],check=True)
MPY=f'{MUSE_ENV}/bin/python'
subprocess.run(['uv','pip','install','--python',MPY,'torch==2.0.1','torchvision==0.15.2','torchaudio==2.0.2','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
subprocess.run(['uv','pip','install','--python',MPY,'-r',str(MUSE/'requirements.txt')],check=True)
subprocess.run(['uv','pip','install','--python',MPY,'openmim'],check=True)
subprocess.run([MPY,'-m','mim','install','mmengine'],check=True)
subprocess.run([MPY,'-m','mim','install','mmcv==2.0.1'],check=True)
subprocess.run([MPY,'-m','mim','install','mmdet==3.1.0'],check=True)
subprocess.run([MPY,'-m','mim','install','mmpose==1.1.0'],check=True)
print('MuseTalk environment installed.')


In [ ]:
MODELS=MUSE/'models'
for d in ['musetalk','musetalkV15','syncnet','dwpose','face-parse-bisent','sd-vae','whisper']: (MODELS/d).mkdir(parents=True,exist_ok=True)
def hf_download(repo_id, local_dir, allow=None):
    from huggingface_hub import snapshot_download
    return snapshot_download(repo_id=repo_id,local_dir=str(local_dir),allow_patterns=allow)
hf_download('TMElyralab/MuseTalk',MODELS, ['musetalkV15/*'])
hf_download('stabilityai/sd-vae-ft-mse',MODELS/'sd-vae')
hf_download('openai/whisper-tiny',MODELS/'whisper')
hf_download('yzd-v/DWPose',MODELS/'dwpose', ['dw-ll_ucoco_384.pth'])
hf_download('ByteDance/LatentSync',MODELS/'syncnet', ['latentsync_syncnet.pt'])
subprocess.run(['uv','pip','install','--python',MPY,'gdown'],check=True)
subprocess.run([MPY,'-m','gdown','154JgKpzCPW82qINcVieuPH3fZ2e0P812','-O',str(MODELS/'face-parse-bisent'/'79999_iter.pth')],check=True)
subprocess.run(['bash','-lc',f"curl -L https://download.pytorch.org/models/resnet18-5c106cde.pth -o '{MODELS/'face-parse-bisent'/'resnet18-5c106cde.pth'}'"],check=True)
print('MuseTalk weights downloaded.')


## Lip-sync the character's dialogue
MuseTalk needs a visible face. If the face is turned away, obscured, very small, or not detected, lip-sync may fail or look unnatural.

In [ ]:
MUSE_INPUT='/content/wan_scene_25fps.mp4'
subprocess.run(['ffmpeg','-y','-i',WAN_VIDEO,'-vf','fps=25','-an','-c:v','libx264','-pix_fmt','yuv420p',MUSE_INPUT],check=True)
cfg=MUSE/'configs/inference/colab_scene.yaml'
cfg.write_text('scene:\n  video_path: '+MUSE_INPUT+'\n  audio_path: /content/dialogue.mp3\n  result_name: character_lipsynced.mp4\n')
MUSE_RESULT=MUSE/'results/test/v15/wan_scene_25fps_dialogue/character_lipsynced.mp4'
cmd=[MPY,'-m','scripts.inference','--inference_config',str(cfg),'--result_dir',str(MUSE/'results/test'),'--unet_model_path',str(MODELS/'musetalkV15/unet.pth'),'--unet_config',str(MODELS/'musetalkV15/musetalk.json'),'--whisper_dir',str(MODELS/'whisper'),'--version','v15','--use_float16','--batch_size','4']
subprocess.run(cmd,cwd=str(MUSE),check=True)
candidates=list((MUSE/'results/test/v15').glob('*/character_lipsynced.mp4'))
if not candidates: candidates=list((MUSE/'results/test/v15').glob('*.mp4'))
if not candidates: raise FileNotFoundError('MuseTalk did not produce a lip-synced MP4; inspect the logs above.')
LIPSYNC_VIDEO=str(candidates[-1])
print('Lip-synced video:',LIPSYNC_VIDEO)


## Mix narration and optional ambience, then export
The dialogue track is already synchronized to the character. Narration and ambience are mixed underneath it.

In [ ]:
FINAL='/content/Wan2.1_Cinematic_Final.mp4'
inputs=['-i',LIPSYNC_VIDEO]
if NARRATION: inputs += ['-i','/content/narration.mp3']
if AMBIENCE_PATH: inputs += ['-stream_loop','-1','-i',AMBIENCE_PATH]
filters=[]; mix_inputs=['[0:a]volume=1.0[dia]']; idx=1
if NARRATION:
    filters.append(f'[{idx}:a]volume=0.8[narr]'); mix_inputs.append('[narr]'); idx+=1
if AMBIENCE_PATH:
    filters.append(f'[{idx}:a]volume=0.18[amb]'); mix_inputs.append('[amb]'); idx+=1
filters.append(''.join(mix_inputs)+f'amix=inputs={len(mix_inputs)}:duration=first:dropout_transition=2, loudnorm[aout]')
cmd=['ffmpeg','-y']+inputs+['-filter_complex',';'.join(filters),'-map','0:v:0','-map','[aout]','-c:v','copy','-c:a','aac','-b:a','192k','-shortest',FINAL]
subprocess.run(cmd,check=True)
print('Final cinematic video ready:',FINAL)
from IPython.display import Video,display
display(Video(FINAL,embed=True))
files.download(FINAL)
